# EquipED DPO Training Template

Before running: open the EquipED admin panel, go to **Training Data**
for the agent you're training, click **Start Training Job**, and paste
the resulting `download_url` and `upload_url` below.

This notebook does not include the actual LoRA/PEFT training loop --
fill that in under the `# TODO` cell based on your chosen base model
and hyperparameters.

In [ ]:
DOWNLOAD_URL = "PASTE_DOWNLOAD_URL_HERE"
UPLOAD_URL = "PASTE_UPLOAD_URL_HERE"

In [ ]:
import io
import json
import zipfile

import requests

response = requests.get(DOWNLOAD_URL)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
    pairs = [json.loads(line) for line in zf.read("pairs.jsonl").decode("utf-8").splitlines() if line]
    manifest = json.loads(zf.read("manifest.json").decode("utf-8"))

print(f"Loaded {len(pairs)} DPO pairs for agent={manifest['agent_id']}")

## Training

TODO: your LoRA/PEFT training loop here. `pairs` is a list of dicts with
`prompt`, `chosen`, `rejected` keys. Produce a trained adapter directory
(e.g. containing `adapter_config.json` and adapter weights), then zip it
before running the push-back cell below.

In [ ]:
# TODO: base model choice, tokenizer, PEFT/LoRA config, training loop.
# When training completes, set ADAPTER_DIR to the directory containing
# the trained adapter's output files.
ADAPTER_DIR = "./trained_adapter"

In [ ]:
import os
import zipfile as zf_module

ADAPTER_ZIP_PATH = "trained_adapter.zip"
with zf_module.ZipFile(ADAPTER_ZIP_PATH, mode="w", compression=zf_module.ZIP_DEFLATED) as zf:
    for root, _dirs, files in os.walk(ADAPTER_DIR):
        for name in files:
            full_path = os.path.join(root, name)
            zf.write(full_path, arcname=name)

with open(ADAPTER_ZIP_PATH, "rb") as f:
    upload_response = requests.post(
        UPLOAD_URL,
        files={"file": ("adapter.zip", f, "application/zip")},
    )
upload_response.raise_for_status()
print("Adapter uploaded:", upload_response.json())